In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os

# 1. SETUP & DATA LOADING
# ---------------------------------------------------------
weights_path = '../data/processed/calibrated_impact_weights.csv'
data_path = '../data/processed/ethiopia_fi_enriched.csv'

# Load weights and historical data
weights = pd.read_csv(weights_path, index_col=0)
df_hist = pd.read_csv(data_path)
df_hist['observation_date'] = pd.to_datetime(df_hist['observation_date'], format='mixed')

# Define Starting Point (2024 Baseline)
latest_access = df_hist[df_hist['indicator_code'] == 'ACC_OWNERSHIP']['value_numeric'].max()
latest_usage = 35.0  # Findex 2024 Baseline for Digital Payments

# Define Future Catalyst Schedule
future_schedule = {
    2025: ['Fayda Digital ID Program Rollout', 'Safaricom Ethiopia Price Increase'],
    2026: ['EthioPay Instant Payment System Launch', 'Foreign Exchange Liberalization'],
    2027: ['M-Pesa EthSwitch Integration']
}

# 2. SCENARIO FORECASTING ENGINE
# ---------------------------------------------------------
def run_scenario(scenario_name='base'):
    """
    Separates Baseline Trend from Event Uplift with Scenario-based uncertainty.
    """
    # Logic for uncertainty treatment
    if scenario_name == 'optimistic':
        nat_growth = 1.5; calib_adj = 1.2  # High policy efficiency
    elif scenario_name == 'pessimistic':
        nat_growth = 0.5; calib_adj = 0.7  # Economic headwinds
    else:
        nat_growth = 1.0; calib_adj = 1.0  # Base Case
    
    results = []
    curr_acc = latest_access
    
    for year in [2024, 2025, 2026, 2027]:
        if year == 2024:
            results.append({'Year': year, 'Baseline_Trend': curr_acc, 'Event_Uplift': 0, 'Total': curr_acc})
            continue
            
        # A. Calculate Baseline (Natural Growth)
        baseline_step = nat_growth
        
        # B. Calculate Event-Driven Uplift (Calibrated)
        year_events = future_schedule.get(year, [])
        event_uplift = 0
        for e in year_events:
            if e in weights.index:
                # Add the weight calibrated in Task 3, adjusted by scenario factor
                event_uplift += weights.loc[e, 'ACC_OWNERSHIP'] * calib_adj
        
        curr_acc += (baseline_step + event_uplift)
        results.append({
            'Year': year, 
            'Baseline_Trend': round(baseline_step, 2), 
            'Event_Uplift': round(event_uplift, 2), 
            'Total': round(curr_acc, 2)
        })
    
    return pd.DataFrame(results)

# Generate Scenarios
df_base = run_scenario('base')
df_opt = run_scenario('optimistic')
df_pess = run_scenario('pessimistic')

# 3. VISUALIZATION (UNCERTAINTY FAN CHART)
# ---------------------------------------------------------
fig = go.Figure()

# Add Confidence/Uncertainty Range (The "Fan")
fig.add_trace(go.Scatter(
    x=df_opt['Year'].tolist() + df_pess['Year'].tolist()[::-1],
    y=df_opt['Total'].tolist() + df_pess['Total'].tolist()[::-1],
    fill='toself',
    fillcolor='rgba(0,176,246,0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip",
    showlegend=True,
    name='Uncertainty Range (Opt/Pess)'
))

# Add Base Forecast Line
fig.add_trace(go.Scatter(
    x=df_base['Year'], y=df_base['Total'],
    mode='lines+markers', name='Base Forecast (Expected)',
    line=dict(color='rgb(0,100,200)', width=4)
))

# Add 60% Target Line
fig.add_hline(y=60, line_dash="dot", line_color="red", 
              annotation_text="NFIS-II 60% Target", annotation_position="bottom right")

fig.update_layout(
    title='Ethiopia Financial Inclusion: 2027 Scenario Forecast',
    xaxis_title='Year', yaxis_title='Account Ownership (%)',
    template='plotly_white', hovermode='x unified'
)

fig.show()

print("📈 Forecast Summary (Base Scenario):")
display(df_base)

📈 Forecast Summary (Base Scenario):


,Year,Baseline_Trend,Event_Uplift,Total
0,2024,70.0,0.0,70.0
1,2025,1.0,1.0,72.0
2,2026,1.0,0.0,73.0
3,2027,1.0,0.0,74.0


In [2]:
# --- Cell 2: Baseline and Future Event Scheduling ---

# 1. Identify Starting Values (The 2024 Baseline)
# We look for the most recent values for our two core dimensions
latest_access = df_hist[df_hist['indicator_code'] == 'ACC_OWNERSHIP']['value_numeric'].max()

# For Usage, we look at Digital Payments or use the 35% baseline if data is sparse
usage_indicator = 'USG_P2P_COUNT' # Using P2P counts as a proxy for Usage growth
latest_usage = 35.0 # Starting point for USG_DIGITAL_PAYMENT based on Global Findex 2024

print(f"📊 2024 Baseline: Access = {latest_access}%, Usage = {latest_usage}%")

# 2. Schedule Future Events (2025 - 2027)
# Based on the National Financial Inclusion Strategy (NFIS-II) timeline
future_schedule = {
    2025: ['Fayda Digital ID Program Rollout', 'Safaricom Ethiopia Price Increase'],
    2026: ['EthioPay Instant Payment System Launch', 'Foreign Exchange Liberalization'],
    2027: ['M-Pesa EthSwitch Integration']
}

# 3. Define Natural Growth (Background trend without major events)
# Ethiopia's natural inclusion growth (historical trend) is roughly 1.0% per year
natural_growth_rate = 1.0 

print("\n✅ Baseline set and future events scheduled.")

📊 2024 Baseline: Access = 70.0%, Usage = 35.0%

✅ Baseline set and future events scheduled.


In [3]:
# --- Cell 3: The Forecasting Engine ---

# 1. Initialize results with the 2024 baseline
forecast_results = [
    {'Year': 2024, 'Access (%)': latest_access, 'Usage (%)': latest_usage, 'Type': 'Actual'}
]

# 2. Tracking variables
current_access = latest_access
current_usage = latest_usage

# 3. Run the Forecast Loop
for year in [2025, 2026, 2027]:
    # A. Natural Trend (Background growth)
    # Access grows naturally at 1%, Usage (digital) grows faster at 2%
    access_lift = 1.0 
    usage_lift = 2.0  
    
    # B. Add Lifts from Scheduled Events
    year_events = future_schedule.get(year, [])
    
    for event_name in year_events:
        if event_name in weights.index:
            # Add weight for Access (Account Ownership)
            access_lift += weights.loc[event_name, 'ACC_OWNERSHIP'] if 'ACC_OWNERSHIP' in weights.columns else 0
            
            # Add weight for Usage (Taking the maximum value among usage indicators)
            # This looks at USG_P2P_COUNT, USG_TELEBIRR_USERS, etc.
            usage_cols = [c for c in weights.columns if c.startswith('USG_')]
            if usage_cols:
                usage_lift += weights.loc[event_name, usage_cols].max()
    
    # Update current totals
    current_access += access_lift
    current_usage += usage_lift
    
    # Store results
    forecast_results.append({
        'Year': year, 
        'Access (%)': round(current_access, 2), 
        'Usage (%)': round(current_usage, 2), 
        'Type': 'Forecast'
    })

# 4. Convert to DataFrame
df_forecast = pd.DataFrame(forecast_results)

print("📈 Forecast Generated for 2025-2027:")
display(df_forecast)

# Save for the Dashboard task
os.makedirs('../data/processed', exist_ok=True)
df_forecast.to_csv('../data/processed/ethiopia_fi_forecast_2027.csv', index=False)

📈 Forecast Generated for 2025-2027:


,Year,Access (%),Usage (%),Type
0,2024,70.0,35.0,Actual
1,2025,72.0,37.0,Forecast
2,2026,73.0,44.0,Forecast
3,2027,74.0,51.0,Forecast


In [4]:
# --- REVISED TASK 4: SCENARIO FORECASTING ENGINE ---

def run_forecast(scenario_type='base'):
    # Adjust parameters based on scenario
    if scenario_type == 'optimistic':
        nat_growth = 1.5; calib_adj = 1.2  # Better policy effectiveness
    elif scenario_type == 'pessimistic':
        nat_growth = 0.5; calib_adj = 0.7  # Economic headwinds
    else:
        nat_growth = 1.0; calib_adj = 1.0  # Base case

    results = []
    curr_acc = latest_access
    
    for year in [2024, 2025, 2026, 2027]:
        if year == 2024:
            results.append({'Year': year, 'Baseline': curr_acc, 'Event_Uplift': 0, 'Total': curr_acc})
            continue
            
        # 1. Calculate Baseline Trend
        baseline_step = nat_growth
        
        # 2. Calculate Event-Driven Uplift
        year_events = future_schedule.get(year, [])
        event_uplift = 0
        for e in year_events:
            if e in weights.index:
                # Apply the calibrated weight adjusted by scenario factor
                event_uplift += weights.loc[e, 'ACC_OWNERSHIP'] * calib_adj
        
        curr_acc += (baseline_step + event_uplift)
        results.append({
            'Year': year, 
            'Baseline_Growth': baseline_step, 
            'Event_Uplift': event_uplift, 
            'Total': round(curr_acc, 2),
            'Scenario': scenario_type
        })
    return pd.DataFrame(results)

# Generate all three scenarios
base_fc = run_forecast('base')
opt_fc = run_forecast('optimistic')
pess_fc = run_forecast('pessimistic')

display(base_fc)

,Year,Baseline,Event_Uplift,Total,Baseline_Growth,Scenario
0,2024,70.0,0.0,70.0,NaN,NaN
1,2025,NaN,1.0,72.0,1.0,base
2,2026,NaN,0.0,73.0,1.0,base
3,2027,NaN,0.0,74.0,1.0,base


In [5]:
# --- Cell 4: Robust Forecasting Visualization ---
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

# 1. Load the forecast data from the CSV we saved in Step 3
forecast_csv_path = '../data/processed/ethiopia_fi_forecast_2027.csv'

if os.path.exists(forecast_csv_path):
    df_plot = pd.read_csv(forecast_csv_path)
    print("✅ Forecast data loaded from CSV.")
else:
    print("❌ Error: Forecast CSV not found. Please re-run Cell 3.")

# 2. Create the figure
fig = go.Figure()

# 3. Add Access Forecast
fig.add_trace(go.Scatter(
    x=df_plot['Year'], 
    y=df_plot['Access (%)'],
    mode='lines+markers',
    name='Access (Account Ownership)',
    line=dict(color='royalblue', width=4),
    marker=dict(size=10)
))

# 4. Add Usage Forecast
fig.add_trace(go.Scatter(
    x=df_plot['Year'], 
    y=df_plot['Usage (%)'],
    mode='lines+markers',
    name='Usage (Digital Payments)',
    line=dict(color='firebrick', width=4, dash='dash'),
    marker=dict(size=10)
))

# 5. Add the 60% Policy Target Line
fig.add_hline(
    y=60, 
    line_dash="dot",
    annotation_text="NFIS-II Target (60%)", 
    annotation_position="bottom right",
    line_color="green",
    line_width=2
)

# 6. Add Scenario Shading
fig.add_vrect(
    x0=2024.5, x1=2027.5,
    fillcolor="gray", opacity=0.1,
    layer="below", line_width=0,
    annotation_text="FORECAST PERIOD", annotation_position="top left"
)

# 7. Styling
fig.update_layout(
    title='Ethiopia Financial Inclusion Forecast: 2025 - 2027',
    xaxis=dict(tickmode='linear', tick0=2024, dtick=1),
    yaxis_title='Percentage of Adult Population (%)',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

# Show the interactive chart
fig.show()

# (Optional) Try to save, but catch error if Kaleido is missing
try:
    fig.write_image("../reports/figures/inclusion_forecast_2027.png")
    print("✅ Image saved to reports/figures/")
except Exception:
    print("⚠️ Note: Image not saved (Kaleido missing), but chart is displayed above.")

✅ Forecast data loaded from CSV.


✅ Image saved to reports/figures/


In [6]:
fig = go.Figure()

# Plot Scenario Range (Shading)
fig.add_trace(go.Scatter(
    x=opt_fc['Year'].tolist() + pess_fc['Year'].tolist()[::-1],
    y=opt_fc['Total'].tolist() + pess_fc['Total'].tolist()[::-1],
    fill='toself', fillcolor='rgba(0,100,80,0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Uncertainty Range (Opt/Pess)'
))

# Plot Base Forecast
fig.add_trace(go.Scatter(x=base_fc['Year'], y=base_fc['Total'], name='Base Forecast', line=dict(color='blue', width=4)))

# Add Target Line
fig.add_hline(y=60, line_dash="dot", line_color="red", annotation_text="60% NFIS-II Target")

fig.update_layout(title="Ethiopia FI Forecast: Scenario Analysis (2025-2027)", template="plotly_white")
fig.show()